# Fase 4 — Las 10 operaciones con Polars

**Proyecto:** BigData-Proy_Ciberseguridad · CIC-IoT-2023  
**Fase:** 4 — Procesamiento distribuido  
**Motor:** Polars

**Polars** es el motor local de la comparación. Lee el CSV completo en memoria y resuelve las transformaciones de forma vectorizada con su motor escrito en Rust, sin centrada en un clúster.

| Dato | Valor |
|---|---|
| Dataset | `01_fase1_datos/muestra/CICIoT2023_sample_600k.csv` |
| Registros | 600,000 |
| Columnas | 40 (39 numéricas + `Label`) |
| Clases | 34 (1 benigna + 33 de ataque) |
| Modo de ejecución | local, en la computadora del proyecto |
| Salida | `results/fase4/Polars/` |

Este notebook ejecuta las 10 operaciones del enunciado con **Polars** y guarda
cada resultado como CSV para poder compararlo con los demás motores.

## Las 10 operaciones del enunciado

| # | Operación | Requisito del enunciado | Archivo de salida |
|---|---|---|---|
| 01 | Carga y validación del dataset | Validación | `01_validacion.csv` |
| 02 | Limpieza de valores no válidos | Limpieza de datos | `02_limpieza.csv` |
| 03 | Tratamiento de duplicados | Eliminación de duplicados | `03_duplicados.csv` |
| 04 | Transformación de variables | Transformación de variables | `04_transformacion_variables.csv` |
| 05 | Filtrado de tráfico | Filtrado | `05_filtrado.csv` |
| 06 | Agregaciones globales | Agregaciones | `06_agregaciones.csv` |
| 07 | Agrupaciones por clase y protocolo | Agrupaciones | `07_agrupaciones.csv` |
| 08 | Ordenamiento y Top-10 | Ordenamiento | `08_ordenamiento_top10.csv` |
| 09 | Métricas de ciberseguridad | Cálculo de métricas | `09_metricas_ciberseguridad.csv` |
| 10 | Resumen consolidado por clase | CRUD y tablas de resultado | `10_resumen_consolidado.csv` |

## Reglas comunes a los cuatro motores

Para que la comparación sea justa, los cuatro notebooks aplican exactamente las
mismas reglas:

1. **Redondeo de ingesta.** Cada motor usa un parser de CSV distinto y los últimos
   decimales de un float pueden diferir en torno a `1e-11`. Todos redondean a
   **6 decimales** al leer el archivo.
2. **Valididad.** Una celda es *no válida* si está vacía, es `NaN` o es infinita.
   La operación 02 elimina las filas que contienen alguna celda no válida.
3. **Extremos.** `minimo_valido` y `maximo_valido` se calculan solo sobre valores
   finitos.
4. **Umbrales.** Los percentiles salen de `04_fase4_procesamiento/comun/umbrales.json`,
   calculado una vez con la biblioteca estándar, para que el filtro sea idéntico.
5. **Escritura.** Todos los CSV se escriben con el mismo formateador
   (`comun/io_comun.py`), así que se comparan celda por celda.

## Preparación del entorno

In [1]:
from __future__ import annotations

import sys
import time
from importlib.metadata import version
from pathlib import Path

AQUI = Path.cwd()
FASE = AQUI if (AQUI / "comun").exists() else AQUI.parent
if str(FASE) not in sys.path:
    sys.path.insert(0, str(FASE))

from comun import config
from comun.io_comun import escribir_csv, escribir_json, pct

import polars as pl

MOTOR = "polars"
VERSION = version("polars")
CSV_MUESTRA = config.CSV_MUESTRA
SALIDA = config.RUTA_RESULTADOS / MOTOR
SALIDA.mkdir(parents=True, exist_ok=True)
UMBRALES = config.cargar_umbrales()
TIEMPOS: list[dict] = []

print(f"Polars      {VERSION}")
print(f"Python      {sys.version.split()[0]}")
print(f"Dataset     {CSV_MUESTRA.name} ({CSV_MUESTRA.stat().st_size:,} bytes)")
print(f"Resultados  {SALIDA}")


def medir(id_operacion: str, nombre: str, funcion):
    """Ejecuta la operación, cronometra y registra el tiempo."""
    inicio = time.perf_counter()
    resultado = funcion()
    segundos = round(time.perf_counter() - inicio, 3)
    TIEMPOS.append(
        {
            "motor": MOTOR,
            "id_operacion": id_operacion,
            "operacion": nombre,
            "segundos": segundos,
        }
    )
    print(f"  -> [{id_operacion}] {nombre}: {segundos:.3f} s")
    return resultado


def csv(nombre: str, columnas: list[str], registros) -> None:
    """Escribe un resultado con el formateador común a los cuatro motores."""
    escribir_csv(SALIDA / nombre, columnas, registros)


def json_(nombre: str, contenido) -> None:
    escribir_json(SALIDA / nombre, contenido)

Polars      1.44.2
Python      3.12.3
Dataset     CICIoT2023_sample_600k.csv (124,498,206 bytes)
Resultados  C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\polars


## Carga del dataset

Leemos el CSV y redondeamos las columnas decimales a 6 decimales, que es la primera regla común: los parsers de cada motor difieren en los últimos decimales de un float.

In [2]:
def cargar() -> pl.DataFrame:
    """Lee el CSV y normaliza la precisión de las columnas decimales."""
    datos = pl.read_csv(CSV_MUESTRA)
    decimales = [c for c, tipo in datos.schema.items() if tipo == pl.Float64]
    return datos.with_columns([pl.col(c).round(config.REDONDEO_INGESTA) for c in decimales])


df = medir("00", "Carga del dataset", cargar)
print(f"Filas: {df.height:,}   Columnas: {df.width}")
df.head(5)

  -> [00] Carga del dataset: 0.238 s
Filas: 600,000   Columnas: 40


Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,cwr_flag_number,ack_count,syn_count,fin_count,rst_count,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ARP,ICMP,IGMP,IPv,LLC,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label
f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,f64,f64,f64,f64,i64,f64,str
0.0,47,64.0,2109.927612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,57800,578,578,578.0,0.0,578.0,0.000475,100,0.0,"""Mirai-greip_flood"""
0.0,47,64.0,5564.803906,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,57800,578,578,578.0,0.0,578.0,0.000181,100,0.0,"""Mirai-greip_flood"""
0.16,47,65.91,1789.707156,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,0.02,0.0,0.01,0.0,0.0,0.99,0.99,56464,60,578,564.64,77.409383,564.64,0.00056,100,5992.212525,"""Mirai-greip_flood"""
0.0,47,64.0,2923.022886,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,57800,578,578,578.0,0.0,578.0,0.0004,100,0.0,"""Mirai-greip_flood"""
0.32,47,63.36,600.243572,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0.04,0.0,0.0,0.0,0.0,0.0,0.04,0.0,0.03,0.0,0.0,0.97,0.97,54588,60,578,545.88,119.524502,545.88,0.001667,100,14286.106667,"""Mirai-greip_flood"""


## Operación 01 — Carga y validación del dataset

**Requisito del enunciado:** Validación

Recorremos las 40 columnas y reportamos el tipo normalizado, cuántas celdas no son válidas (vacías, `NaN` o infinitas), cuántos valores distintos tiene y su rango considerando solo valores finitos.

In [3]:
def _tipo(dtype) -> str:
    """Tipo normalizado para que los cuatro motores coincidan."""
    if dtype == pl.String:
        return "texto"
    if dtype.is_integer():
        return "entero"
    if dtype.is_float():
        return "decimal"
    return "otro"


def _no_validas(serie, tipo: str) -> int:
    """Celdas vacías, NaN o infinitas."""
    invalidas = serie.null_count()
    if tipo == "decimal":
        invalidas = invalidas + serie.is_nan().sum() + serie.is_infinite().sum()
    return int(invalidas)


def _rango_valido(serie, tipo: str):
    """Mínimo y máximo sobre valores finitos."""
    if tipo == "texto":
        return "", ""
    if tipo == "decimal":
        serie = serie.filter(serie.is_finite())
    return serie.min(), serie.max()


def op01():
    registros = []
    for columna in config.COLUMNAS:
        serie = df[columna]
        tipo = _tipo(df.schema[columna])
        invalidas = _no_validas(serie, tipo)
        minimo, maximo = _rango_valido(serie, tipo)
        registros.append(
            {
                "columna": columna,
                "tipo": tipo,
                "celdas_no_validas": invalidas,
                "pct_no_validas": round(pct(invalidas, df.height), 6),
                "distintos": serie.n_unique() if columna in config.COLS_DISTINTOS else "",
                "minimo_valido": minimo,
                "maximo_valido": maximo,
            }
        )

    csv(
        "01_validacion.csv",
        ["columna", "tipo", "celdas_no_validas", "pct_no_validas", "distintos",
         "minimo_valido", "maximo_valido"],
        registros,
    )
    json_(
        "01_validacion_resumen.json",
        {
            "motor": MOTOR,
            "version": VERSION,
            "filas": df.height,
            "columnas": len(config.COLUMNAS),
            "redondeo_decimales_ingesta": config.REDONDEO_INGESTA,
            "nulos_totales": int(sum(df.null_count().row(0))),
            "infinitos_totales": int(
                sum(int(df[c].is_infinite().sum()) for c in config.COLUMNAS
                    if _tipo(df.schema[c]) == "decimal")
            ),
            "etiquetas_distintas": df["Label"].n_unique(),
        },
    )
    return registros


validacion = medir("01", "Carga y validación del dataset", op01)
pl.DataFrame(validacion).head(12)

  -> [01] Carga y validación del dataset: 0.247 s


columna,tipo,celdas_no_validas,pct_no_validas,distintos,minimo_valido,maximo_valido
str,str,i64,f64,str,str,str
"""Header_Length""","""decimal""",0,0.0,"""""","""0""","""60"""
"""Protocol Type""","""entero""",0,0.0,"""5""","""0""","""47"""
"""Time_To_Live""","""decimal""",0,0.0,"""""","""0""","""255"""
"""Rate""","""decimal""",13,0.002167,"""99172""","""0.000428""","""7340032"""
"""fin_flag_number""","""decimal""",0,0.0,"""110""","""0""","""1"""
…,…,…,…,…,…,…
"""psh_flag_number""","""decimal""",0,0.0,"""143""","""0""","""1"""
"""ack_flag_number""","""decimal""",0,0.0,"""188""","""0""","""1"""
"""ece_flag_number""","""decimal""",0,0.0,"""11""","""0""","""0.6"""


## Operación 02 — Limpieza de valores no válidos

**Requisito del enunciado:** Limpieza de datos

Regla común: una celda no válida (vacía, `NaN` o infinita) pasa a `null` y la fila se elimina. En esta muestra se detectan celdas vacías en `Std` y `Variance`, y 13 valores infinitos en `Rate`.

In [4]:
def op02():
    tipos = {c: _tipo(df.schema[c]) for c in config.COLUMNAS}
    antes = len(df)
    no_validas_antes = {c: _no_validas(df[c], tipos[c]) for c in config.COLUMNAS}
    decimales = [c for c in config.COLUMNAS if tipos[c] == "decimal"]

    # Cualquier valor no finito se convierte en nulo.
    saneado = df.with_columns(
        [
            pl.when(pl.col(c).is_finite()).then(pl.col(c)).otherwise(None).alias(c)
            for c in decimales
        ]
    )
    limpio = saneado.drop_nulls()
    despues = len(limpio)
    no_validas_despues = {c: _no_validas(limpio[c], tipos[c]) for c in config.COLUMNAS}

    registros = [
        {
            "columna": c,
            "valores_no_validos_antes": no_validas_antes[c],
            "valores_no_validos_despues": no_validas_despues[c],
            "filas_eliminadas": antes - despues,
        }
        for c in config.COLUMNAS
        if no_validas_antes[c] > 0
    ]
    csv("02_limpieza.csv",
        ["columna", "valores_no_validos_antes", "valores_no_validos_despues",
         "filas_eliminadas"],
        registros)
    json_(
        "02_limpieza_resumen.json",
        {
            "motor": MOTOR,
            "criterio": "se elimina la fila con una celda vacía, NaN o infinita",
            "filas_antes": antes,
            "filas_despues": despues,
            "filas_eliminadas": antes - despues,
            "columnas_afectadas": registros,
        },
    )
    return limpio


limpio = medir("02", "Limpieza de valores no válidos", op02)
print(f"Filas antes: {len(df):,}   Filas después: {len(limpio):,}   Eliminadas: {len(df) - len(limpio):,}")

  -> [02] Limpieza de valores no válidos: 0.122 s
Filas antes: 600,000   Filas después: 599,987   Eliminadas: 13


## Operación 03 — Tratamiento de duplicados

**Requisito del enunciado:** Eliminación de duplicados

Medimos los duplicados con dos criterios: la fila completa (las 40 columnas) y un conjunto de claves (`Label`, `Protocol Type`, `Tot size`, `IAT`, `Rate`, `Number`). Además mostramos los 10 grupos más repetidos.

In [5]:
COLS_CLAVE = config.COLS_CLAVE_DUPLICADOS


def op03():
    antes = len(limpio)
    exactos = limpio.unique()
    por_clave = limpio.unique(subset=COLS_CLAVE)

    registros = [
        {
            "criterio": "filas_completas",
            "columnas_clave": f"las {len(config.COLUMNAS)} columnas",
            "filas_antes": antes,
            "filas_despues": len(exactos),
            "duplicados_eliminados": antes - len(exactos),
            "pct_duplicados": round(pct(antes - len(exactos), antes), 6),
        },
        {
            "criterio": "columnas_clave",
            "columnas_clave": ", ".join(COLS_CLAVE),
            "filas_antes": antes,
            "filas_despues": len(por_clave),
            "duplicados_eliminados": antes - len(por_clave),
            "pct_duplicados": round(pct(antes - len(por_clave), antes), 6),
        },
    ]
    csv("03_duplicados.csv",
        ["criterio", "columnas_clave", "filas_antes", "filas_despues",
         "duplicados_eliminados", "pct_duplicados"],
        registros)

    ejemplos = (
        limpio.group_by(COLS_CLAVE)
        .len()
        .rename({"len": "apariciones"})
        .sort(
            ["apariciones", *COLS_CLAVE],
            descending=[True, *([False] * len(COLS_CLAVE))],
        )
        .head(config.TOP_N)
    )
    csv("03_duplicados_ejemplos.csv",
        ["apariciones"] + COLS_CLAVE,
        ejemplos.to_dicts())
    return registros


duplicados = medir("03", "Tratamiento de duplicados", op03)
pl.DataFrame(duplicados)

  -> [03] Tratamiento de duplicados: 0.818 s


criterio,columnas_clave,filas_antes,filas_despues,duplicados_eliminados,pct_duplicados
str,str,i64,i64,i64,f64
"""filas_completas""","""las 40 columnas""",599987,395496,204491,34.082572
"""columnas_clave""","""Label, Protocol Type, Tot size…",599987,345902,254085,42.348418


## Operación 04 — Transformación de variables

**Requisito del enunciado:** Transformación de variables

Creamos seis variables nuevas a partir de las existentes: tamaño en kilobytes, tasa en Mbps, coeficiente de variación, suma de flags TCP, protocolo principal leído de las columnas one-hot y una clasificación del IAT con los percentiles 50 y 95.

In [6]:
COLS_NUEVAS = [
    "size_kb",
    "rate_mbps",
    "coef_variacion",
    "total_flags",
    "protocolo_principal",
    "rango_iat",
]

DEFINICIONES = {
    "size_kb": "Tot size / 1024 (kilobytes)",
    "rate_mbps": "Rate / 1 000 000 (paquetes por segundo)",
    "coef_variacion": "Std / AVG (dispersion del tamaño de paquete)",
    "total_flags": "Suma de los siete indicadores de flags TCP",
    "protocolo_principal": "Protocolo con valor 1 en las columnas one-hot",
    "rango_iat": "IAT clasificado con los percentiles 50 y 95: bajo, medio, alto",
}

In [7]:
def op04():
    iat = UMBRALES["IAT"]

    # El protocolo principal es la primera columna one-hot con valor 1.
    protocolo = pl.lit("otro")
    for columna in reversed(config.COLS_PROTOCOLO):
        protocolo = (
            pl.when(pl.col(columna) > 0).then(pl.lit(columna)).otherwise(protocolo)
        )

    transformado = limpio.with_columns(
        (pl.col("Tot size") / 1024).alias("size_kb"),
        (pl.col("Rate") / 1_000_000).alias("rate_mbps"),
        pl.when(pl.col("AVG") > 0)
        .then(pl.col("Std") / pl.col("AVG"))
        .otherwise(None)
        .alias("coef_variacion"),
        pl.sum_horizontal([pl.col(c) for c in config.COLS_FLAGS]).alias("total_flags"),
        protocolo.alias("protocolo_principal"),
        pl.when(pl.col("IAT") <= iat["p50"])
        .then(pl.lit("bajo"))
        .when(pl.col("IAT") <= iat["p95"])
        .then(pl.lit("medio"))
        .otherwise(pl.lit("alto"))
        .alias("rango_iat"),
    )

    registros = []
    for columna in COLS_NUEVAS:
        serie = transformado[columna]
        texto = serie.dtype == pl.String
        registros.append(
            {
                "columna": columna,
                "tipo": _tipo(serie.dtype),
                "nulos": int(serie.null_count()),
                "minimo": "" if texto else serie.min(),
                "maximo": "" if texto else serie.max(),
                "media": "" if texto else serie.mean(),
                "distintos": serie.n_unique() if texto else "",
                "definicion": DEFINICIONES[columna],
            }
        )
    csv("04_transformacion_variables.csv",
        ["columna", "tipo", "nulos", "minimo", "maximo", "media", "distintos", "definicion"],
        registros)

    muestra = transformado.select(
        ["Label", "Protocol Type", "Rate", "Tot size", "IAT", *COLS_NUEVAS]
    ).head(config.FILAS_MUESTRA_TRANSFORMACION)
    csv("04_transformacion_muestra.csv", muestra.columns, muestra.to_dicts())
    return transformado, registros


transformado, variables = medir("04", "Transformación de variables", op04)
pl.DataFrame(variables)

  -> [04] Transformación de variables: 0.077 s


columna,tipo,nulos,minimo,maximo,media,distintos,definicion
str,str,i64,str,str,str,str,str
"""size_kb""","""decimal""",0,"""0.044921875""","""4.64560546875""","""0.12845749221506894""","""""","""Tot size / 1024 (kilobytes)"""
"""rate_mbps""","""decimal""",0,"""0.00000000042799999999999997""","""7.340032""","""0.02851448277422539""","""""","""Rate / 1 000 000 (paquetes por…"
"""coef_variacion""","""decimal""",0,"""0""","""5.916700694160882""","""0.10585179549867402""","""""","""Std / AVG (dispersion del tama…"
"""total_flags""","""decimal""",0,"""0""","""2.38""","""0.6131944215224661""","""""","""Suma de los siete indicadores …"
"""protocolo_principal""","""texto""",0,"""""","""""","""""","""13""","""Protocolo con valor 1 en las c…"
"""rango_iat""","""texto""",0,"""""","""""","""""","""3""","""IAT clasificado con los percen…"


In [8]:
# A partir de aqui df pasa a ser el dataset limpio y transformado:
# las operaciones 05 a 10 ya pueden usar las columnas nuevas.
df = transformado
print(f"Columnas del dataset transformado: {len(df.columns)}")

Columnas del dataset transformado: 46


## Operación 05 — Filtrado de tráfico

**Requisito del enunciado:** Filtrado

Aplicamos cuatro filtros con umbrales tomados del percentil 95 y 50 de `Rate`, `Tot size` e `IAT`. Los percentiles se calcularon una vez con la biblioteca estándar y están en `comun/umbrales.json`, así que los cuatro motores filtran exactamente lo mismo.

In [9]:
def op05():
    total = len(df)
    rate, size, iat = UMBRALES["Rate"], UMBRALES["Tot size"], UMBRALES["IAT"]

    filtros = [
        ("trafico_alto", f"Rate >= p95 ({rate['p95']:.6f})", rate["p95"],
         df["Rate"] >= rate["p95"]),
        ("paquetes_grandes", f"Tot size >= p95 ({size['p95']:.6f})", size["p95"],
         df["Tot size"] >= size["p95"]),
        ("trafico_intenso",
         f"Rate >= p95 ({rate['p95']:.6f}) y Tot size >= p50 ({size['p50']:.6f})",
         rate["p95"],
         (df["Rate"] >= rate["p95"]) & (df["Tot size"] >= size["p50"])),
        ("iat_reducido", f"0 < IAT <= p95 ({iat['p95']:.6f})", iat["p95"],
         (df["IAT"] > 0) & (df["IAT"] <= iat["p95"])),
    ]

    registros = []
    for nombre, condicion, umbral, mascara in filtros:
        encontradas = int(mascara.sum())
        registros.append(
            {
                "filtro": nombre,
                "condicion": condicion,
                "umbral": umbral,
                "filas_encontradas": encontradas,
                "pct_del_total": round(pct(encontradas, total), 6),
            }
        )
    csv("05_filtrado.csv",
        ["filtro", "condicion", "umbral", "filas_encontradas", "pct_del_total"],
        registros)

    intenso = df.filter(
        (df["Rate"] >= rate["p95"]) & (df["Tot size"] >= size["p50"])
    )
    por_label = (
        intenso.group_by("Label")
        .agg(
            pl.len().alias("registros"),
            pl.col("Tot size").sum().alias("volumen_bytes"),
            pl.col("Rate").mean().alias("media_rate"),
        )
        .sort(["registros", "Label"], descending=[True, False])
    )
    csv("05_filtrado_por_label.csv",
        ["Label", "registros", "volumen_bytes", "media_rate"],
        por_label.to_dicts())
    return registros


filtrado = medir("05", "Filtrado de tráfico", op05)
pl.DataFrame(filtrado)

  -> [05] Filtrado de tráfico: 0.050 s


filtro,condicion,umbral,filas_encontradas,pct_del_total
str,str,f64,i64,f64
"""trafico_alto""","""Rate >= p95 (63492.340297)""",63492.340297,30029,5.004942
"""paquetes_grandes""","""Tot size >= p95 (586.680000)""",586.68,31235,5.205946
"""trafico_intenso""","""Rate >= p95 (63492.340297) y T…",63492.340297,30013,5.002275
"""iat_reducido""","""0 < IAT <= p95 (0.001070)""",0.00107,569989,95.000225


## Operación 06 — Agregaciones globales

**Requisito del enunciado:** Agregaciones

Siete métricas (conteo, suma, media, mediana, mínimo, máximo y desviación estándar) sobre las diez columnas indicadoras. La respuesta se guarda en formato largo: una fila por combinación columna-métrica.

In [10]:
def op06():
    registros = []
    for columna in config.COLS_INDICADORES:
        serie = df[columna]
        for metrica, valor in (
            ("conteo", serie.count()),
            ("suma", serie.sum()),
            ("media", serie.mean()),
            ("mediana", serie.median()),
            ("minimo", serie.min()),
            ("maximo", serie.max()),
            ("desviacion_estandar", serie.std()),
        ):
            registros.append({"columna": columna, "metrica": metrica, "valor": valor})
    csv("06_agregaciones.csv", ["columna", "metrica", "valor"], registros)
    return registros


agregaciones = medir("06", "Agregaciones globales", op06)
pl.DataFrame(agregaciones).head(14)

  -> [06] Agregaciones globales: 0.064 s


columna,metrica,valor
str,str,f64
"""Rate""","""conteo""",599987.0
"""Rate""","""suma""",1.7108e10
"""Rate""","""media""",28514.482774
"""Rate""","""mediana""",24662.221438
"""Rate""","""minimo""",0.000428
…,…,…
"""Tot size""","""media""",131.540472
"""Tot size""","""mediana""",60.0
"""Tot size""","""minimo""",46.0


## Operación 07 — Agrupaciones por clase y protocolo

**Requisito del enunciado:** Agrupaciones

Agrupamos por `Label` y `Protocol Type`, y de cada grupo calculamos registros, porcentaje sobre el total, volumen, media del tamaño, media de la tasa y media del IAT.

In [11]:
def op07():
    total = len(df)
    grupos = (
        df.group_by(["Label", "Protocol Type"])
        .agg(
            pl.len().alias("registros"),
            pl.col("Tot size").sum().alias("volumen_bytes"),
            pl.col("Tot size").mean().alias("media_tot_size"),
            pl.col("Rate").mean().alias("media_rate"),
            pl.col("IAT").mean().alias("media_iat"),
        )
        .sort(["Label", "Protocol Type"])
    )
    registros = [
        {
            "Label": f[0],
            "Protocol Type": f[1],
            "registros": f[2],
            "pct_registros": round(pct(f[2], total), 6),
            "volumen_bytes": f[3],
            "media_tot_size": f[4],
            "media_rate": f[5],
            "media_iat": f[6],
        }
        for f in grupos.rows()
    ]
    csv("07_agrupaciones.csv",
        ["Label", "Protocol Type", "registros", "pct_registros", "volumen_bytes",
         "media_tot_size", "media_rate", "media_iat"],
        registros)
    return registros


agrupaciones = medir("07", "Agrupaciones por clase y protocolo", op07)
pl.DataFrame(agrupaciones).head(10)

  -> [07] Agrupaciones por clase y protocolo: 0.024 s


Label,Protocol Type,registros,pct_registros,volumen_bytes,media_tot_size,media_rate,media_iat
str,i64,i64,f64,f64,f64,f64,f64
"""Backdoor_Malware""",6,28,0.004667,10134.8,361.957143,673.427166,0.026105
"""Backdoor_Malware""",17,14,0.002333,2075.0,148.214286,48.065148,0.038464
"""Benign""",0,31,0.005167,4701.8,151.670968,221.481694,0.019105
"""Benign""",1,3,0.0005,408.8,136.266667,128.237094,0.018639
"""Benign""",6,12885,2.147547,8.3924e6,651.327377,2901.537006,0.00874
"""Benign""",17,1166,0.194338,178502.4,153.089537,152.587589,0.013828
"""BrowserHijacking""",6,67,0.011167,39705.2,592.614925,23350.552569,0.015639
"""BrowserHijacking""",17,9,0.0015,2711.4,301.266667,259.118631,0.014353
"""CommandInjection""",6,54,0.009,33903.5,627.842593,2927.71713,0.01549


## Operación 08 — Ordenamiento y Top-10

**Requisito del enunciado:** Ordenamiento

Ordenamos las 34 clases por volumen de tráfico descendente y conservamos las diez primeras, que concentran la mayor parte de los bytes del dataset.

In [12]:
def _resumen_por_clase() -> pl.DataFrame:
    return (
        df.group_by("Label")
        .agg(
            pl.len().alias("registros"),
            pl.col("Tot size").sum().alias("volumen_bytes"),
            pl.col("Rate").mean().alias("media_rate"),
        )
        .sort(["volumen_bytes", "Label"], descending=[True, False])
    )


def op08():
    volumen_total = float(df["Tot size"].sum())
    top = _resumen_por_clase().head(config.TOP_N)
    registros = [
        {
            "posicion": posicion,
            "Label": f[0],
            "registros": f[1],
            "volumen_bytes": f[2],
            "volumen_mb": f[2] / (1024 * 1024),
            "pct_volumen": round(pct(float(f[2]), volumen_total), 6),
            "media_rate": f[3],
        }
        for posicion, f in enumerate(top.rows(), start=1)
    ]
    csv("08_ordenamiento_top10.csv",
        ["posicion", "Label", "registros", "volumen_bytes", "volumen_mb",
         "pct_volumen", "media_rate"],
        registros)
    return registros


top10 = medir("08", "Ordenamiento y Top-10", op08)
pl.DataFrame(top10)

  -> [08] Ordenamiento y Top-10: 0.043 s


posicion,Label,registros,volumen_bytes,volumen_mb,pct_volumen,media_rate
i64,str,i64,f64,f64,f64,f64
1,"""Benign""",14085,8.5760e6,8.178679,10.866303,2667.481157
2,"""Mirai-greeth_flood""",12719,7.4334e6,7.089085,9.418655,5576.009272
3,"""Mirai-udpplain""",11423,6.2437e6,5.954485,7.911209,6132.884517
4,"""DDoS-ICMP_Flood""",92356,5.6100e6,5.350113,7.108233,39957.267678
5,"""Mirai-greip_flood""",9642,5.4520e6,5.199396,6.907988,5055.603642
6,"""DDoS-ICMP_Fragmentation""",5804,5.1405e6,4.902337,6.513311,2973.919964
7,"""DDoS-UDP_Flood""",69419,4.2186e6,4.023201,5.345279,33345.35526
8,"""DDoS-TCP_Flood""",57687,3.6317e6,3.463455,4.601593,32271.151585
9,"""DoS-UDP_Flood""",39414,3.5168e6,3.353916,4.456058,20014.109743


## Operación 09 — Métricas de ciberseguridad

**Requisito del enunciado:** Cálculo de métricas

Indicadores de valor para la toma de decisiones: población de ataques y tráfico benigno, volumen, comportamiento de los flags TCP, concentración del ataque y calidad de los datos.

In [13]:
def op09():
    total = len(df)
    benignos = int((df["Label"] == config.ETIQUETA_BENIGNA).sum())
    ataques = total - benignos
    volumen_total = float(df["Tot size"].sum())
    paquetes = float(df["Number"].sum())
    clases = int(df["Label"].n_unique())
    clases_ataque = int(
        df.filter(pl.col("Label") != config.ETIQUETA_BENIGNA)["Label"].n_unique()
    )
    por_clase = _resumen_por_clase()
    principal = por_clase.row(0, named=True)
    top5 = por_clase.head(5)
    media_syn = float(df["syn_flag_number"].mean())
    media_ack = float(df["ack_flag_number"].mean())
    duplicados = total - df.unique().height

    metricas = [
        ("poblacion", "total_registros", total),
        ("poblacion", "total_clases", clases),
        ("poblacion", "clases_de_ataque", clases_ataque),
        ("poblacion", "registros_benignos", benignos),
        ("poblacion", "registros_de_ataque", ataques),
        ("poblacion", "pct_registros_benignos", round(pct(benignos, total), 6)),
        ("poblacion", "pct_registros_ataque", round(pct(ataques, total), 6)),
        ("volumen", "volumen_total_bytes", volumen_total),
        ("volumen", "volumen_total_mb", volumen_total / (1024 * 1024)),
        ("volumen", "paquetes_totales", paquetes),
        ("volumen", "tamano_medio_paquete_bytes",
         volumen_total / paquetes if paquetes else 0.0),
        ("volumen", "tasa_media", float(df["Rate"].mean())),
        ("comportamiento", "media_syn_flag", media_syn),
        ("comportamiento", "media_ack_flag", media_ack),
        ("comportamiento", "media_rst_flag", float(df["rst_flag_number"].mean())),
        ("comportamiento", "media_fin_flag", float(df["fin_flag_number"].mean())),
        ("comportamiento", "ratio_syn_ack", media_syn / media_ack if media_ack else 0.0),
        ("comportamiento", "iat_medio", float(df["IAT"].mean())),
        ("concentracion", "clase_principal", principal["Label"]),
        ("concentracion", "pct_registros_clase_principal",
         round(pct(principal["registros"], total), 6)),
        ("concentracion", "pct_registros_top5",
         round(pct(int(top5["registros"].sum()), total), 6)),
        ("concentracion", "pct_volumen_top5",
         round(pct(float(top5["volumen_bytes"].sum()), volumen_total), 6)),
        ("calidad", "filas_duplicadas", duplicados),
        ("calidad", "pct_filas_duplicadas", round(pct(duplicados, total), 6)),
        ("calidad", "pct_filas_unicas", round(pct(total - duplicados, total), 6)),
    ]
    registros = [
        {"categoria": c, "metrica": m, "valor": v} for c, m, v in metricas
    ]
    csv("09_metricas_ciberseguridad.csv", ["categoria", "metrica", "valor"], registros)
    return registros


metricas = medir("09", "Métricas de ciberseguridad", op09)
pl.DataFrame(metricas)

  -> [09] Métricas de ciberseguridad: 0.486 s


categoria,metrica,valor
str,str,str
"""poblacion""","""total_registros""","""599987"""
"""poblacion""","""total_clases""","""34"""
"""poblacion""","""clases_de_ataque""","""33"""
"""poblacion""","""registros_benignos""","""14085"""
"""poblacion""","""registros_de_ataque""","""585902"""
…,…,…
"""concentracion""","""pct_registros_top5""","""23.37134"""
"""concentracion""","""pct_volumen_top5""","""42.212388"""
"""calidad""","""filas_duplicadas""","""204491"""


## Operación 10 — Resumen consolidado por clase

**Requisito del enunciado:** CRUD y tablas de resultado

Tabla final: una fila por clase que reúne el conteo, el peso porcentual, el volumen y las medias de las variables transformadas. Es el resultado que se usará en el informe final del proyecto.

In [14]:
def op10():
    total = len(df)
    volumen_total = float(df["Tot size"].sum())
    resumen = (
        df.group_by("Label")
        .agg(
            pl.len().alias("registros"),
            pl.col("Tot size").sum().alias("volumen_bytes"),
            pl.col("Rate").mean().alias("media_rate"),
            pl.col("Tot size").mean().alias("media_tot_size"),
            pl.col("size_kb").mean().alias("media_size_kb"),
            pl.col("IAT").mean().alias("media_iat"),
            pl.col("syn_flag_number").mean().alias("media_syn"),
            pl.col("ack_flag_number").mean().alias("media_ack"),
            pl.col("rst_flag_number").mean().alias("media_rst"),
            pl.col("total_flags").mean().alias("media_total_flags"),
            pl.col("Protocol Type").n_unique().alias("protocolos_distintos"),
        )
        .sort(["registros", "Label"], descending=[True, False])
    )
    registros = [
        {
            "posicion": posicion,
            "Label": f[0],
            "registros": f[1],
            "pct_registros": round(pct(f[1], total), 6),
            "volumen_bytes": f[2],
            "pct_volumen": round(pct(float(f[2]), volumen_total), 6),
            "media_rate": f[3],
            "media_tot_size": f[4],
            "media_size_kb": f[5],
            "media_iat": f[6],
            "media_syn": f[7],
            "media_ack": f[8],
            "media_rst": f[9],
            "media_total_flags": f[10],
            "protocolos_distintos": f[11],
        }
        for posicion, f in enumerate(resumen.rows(), start=1)
    ]
    csv("10_resumen_consolidado.csv",
        ["posicion", "Label", "registros", "pct_registros", "volumen_bytes",
         "pct_volumen", "media_rate", "media_tot_size", "media_size_kb", "media_iat",
         "media_syn", "media_ack", "media_rst", "media_total_flags", "protocolos_distintos"],
        registros)
    return registros


consolidado = medir("10", "Resumen consolidado por clase", op10)
pl.DataFrame(consolidado).head(10)

  -> [10] Resumen consolidado por clase: 0.039 s


posicion,Label,registros,pct_registros,volumen_bytes,pct_volumen,media_rate,media_tot_size,media_size_kb,media_iat,media_syn,media_ack,media_rst,media_total_flags,protocolos_distintos
i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
1,"""DDoS-ICMP_Flood""",92356,15.393,5.6100e6,7.108233,39957.267678,60.743213,0.05932,0.000108,0.00033,0.001629,0.000048,0.002606,4
2,"""DDoS-UDP_Flood""",69419,11.570084,4.2186e6,5.345279,33345.35526,60.770564,0.059346,0.000063,0.000589,0.002523,0.000074,0.004228,4
3,"""DDoS-TCP_Flood""",57687,9.614708,3.6317e6,4.601593,32271.151585,62.955185,0.06148,0.000061,0.000611,0.002596,0.000057,0.0043,2
4,"""DDoS-PSHACK_FLOOD""",52521,8.75369,3.1758e6,4.023902,32611.295328,60.46661,0.059049,0.000056,0.000368,0.967409,0.031412,1.96475,2
5,"""DDoS-SYN_Flood""",52065,8.677688,3.2478e6,4.115232,28803.489591,62.380613,0.060919,0.000071,0.98643,0.016452,0.008175,1.012184,2
6,"""DDoS-RSTFINFLOOD""",51887,8.648021,3.1769e6,4.025337,33311.555872,61.227279,0.059792,0.000366,0.000336,0.00202,0.995544,1.994087,2
7,"""DDoS-SynonymousIP_Flood""",46151,7.692,2.8045e6,3.553462,32035.141131,60.76756,0.059343,0.000062,0.996234,0.00156,0.000068,0.998458,2
8,"""DoS-UDP_Flood""",39414,6.569142,3.5168e6,4.456058,20014.109743,89.228085,0.087137,0.060188,0.000858,0.005655,0.000136,0.008965,3
9,"""DoS-TCP_Flood""",34265,5.710957,2.1625e6,2.74,25445.809057,63.110424,0.061631,0.000708,0.000799,0.009931,0.004834,0.017312,2


## Resumen de la ejecución

La tabla muestra el tiempo de cada operación. El total incluye la carga del CSV.

In [15]:
tabla_tiempos = pl.DataFrame(TIEMPOS)
total = round(sum(fila["segundos"] for fila in TIEMPOS), 3)
tabla_tiempos

csv("00_tiempos.csv", ["motor", "id_operacion", "operacion", "segundos"], TIEMPOS)
json_(
    "00_resumen_ejecucion.json",
    {
        "motor": MOTOR,
        "version": VERSION,
        "dataset": CSV_MUESTRA.name,
        "filas_cargadas": len(limpio),
        "columnas": len(config.COLUMNAS),
        "operaciones": 10,
        "segundos_totales": total,
        "tiempos": TIEMPOS,
    },
)
print(f"Tiempo total de las 10 operaciones: {total:.3f} s")
print(f"Archivos escritos en: {SALIDA}")

Tiempo total de las 10 operaciones: 2.208 s
Archivos escritos en: C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\polars
